# Part 6 - Working RAG Notebook



This notebook demonstrates a complete mini RAG pipeline:

1) Load a PDF

2) Chunk with `RecursiveCharacterTextSplitter` (including chunk-size experiment)

3) Embed + store in ChromaDB

4) Test retrieval with 3 queries and relevance notes


In [1]:
%pip install -q langchain langchain-openai langchain-community langchain-text-splitters chromadb pypdf python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_classic.chains import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    raise ValueError('OPENAI_API_KEY not found. Add it to .env before running.')

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)
print(f'Working directory: {BASE_DIR}')
print(f'Data directory: {DATA_DIR}')

Working directory: c:\Users\leiwang\OneDrive - Stanford Childrens Health\AI-Internship\ai-engineering-bootcamp\rag-vector-databases
Data directory: c:\Users\leiwang\OneDrive - Stanford Childrens Health\AI-Internship\ai-engineering-bootcamp\rag-vector-databases\data


## PDF path

Put your PDF in the `data/` folder (or set `PDF_PATH` below). This notebook only **reads** PDFs via `PyPDFLoader`; it does not generate PDFs.

In [17]:
# Optional: set an explicit file, e.g. PDF_PATH = DATA_DIR / "my_doc.pdf"
PDF_PATH = DATA_DIR

if PDF_PATH.is_dir():
    pdf_files = sorted(PDF_PATH.glob('*.pdf'))
else:
    pdf_files = [PDF_PATH]

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF in {PDF_PATH}. Add a .pdf file there, or set PDF_PATH to your file."
    )

print(f"PDF files found: {len(pdf_files)}")
for pdf_file in pdf_files:
    print(f"- {pdf_file.name}")

PDF files found: 3
- 062024_ChatGPT_for_Job_Seekers_DL.pdf
- Chat-GPT-AI-and-the-Job-Search-Career-Guide-8.5x11.pdf
- chat-gpt-guide.pdf


## Step 1: Load

Use a LangChain document loader and print required diagnostics.

In [20]:
documents = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(str(pdf_file))
    file_documents = loader.load()
    documents.extend(file_documents)
    print(f"Loaded {len(file_documents)} page documents from {pdf_file.name}")

print(f"Total page documents loaded: {len(documents)}")
first_sample = documents[0].page_content[:500] if documents else ''
print('Sample of first document content:')
print(first_sample)

Loaded 2 page documents from 062024_ChatGPT_for_Job_Seekers_DL.pdf
Loaded 2 page documents from Chat-GPT-AI-and-the-Job-Search-Career-Guide-8.5x11.pdf
Loaded 2 page documents from chat-gpt-guide.pdf
Total page documents loaded: 6
Sample of first document content:
Supercharge Your Job Search with AI: Direct Strategies 
By: Daniel Lopez, Career Services Advisor at RochesterWorks 
I recently read an article stating that only 7% of Americans use ChatGPT daily. It's surprising how 
under-adopted such a powerful tool remains. When I teach a class or show an individual how to use 
AI for their job search, the best part is seeing the reactions when people realize just how incredibly 
useful AI can be. It's time for everyone to be aware of what an invaluable reso


## Step 2: Chunk

Use `RecursiveCharacterTextSplitter` and compare at least two chunk sizes.

In [22]:
def chunk_and_report(docs, chunk_size, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(docs)
    lengths = [len(c.page_content) for c in chunks] if chunks else [0]
    print(f'--- chunk_size={chunk_size}, chunk_overlap={chunk_overlap} ---')
    print(f'Total chunks created: {len(chunks)}')
    print(f'Smallest chunk chars: {min(lengths)}')
    print(f'Largest chunk chars: {max(lengths)}')
    return chunks

chunks_500 = chunk_and_report(documents, chunk_size=500, chunk_overlap=100)
chunks_1000 = chunk_and_report(documents, chunk_size=1000, chunk_overlap=100)

print()
print('Experiment notes:')
print('- Smaller chunk_size (500) usually creates MORE chunks with tighter context windows.')
print('- Larger chunk_size (1000) usually creates FEWER chunks with broader context per chunk.')

--- chunk_size=500, chunk_overlap=100 ---
Total chunks created: 26
Smallest chunk chars: 140
Largest chunk chars: 499
--- chunk_size=1000, chunk_overlap=100 ---
Total chunks created: 13
Smallest chunk chars: 203
Largest chunk chars: 980

Experiment notes:
- Smaller chunk_size (500) usually creates MORE chunks with tighter context windows.
- Larger chunk_size (1000) usually creates FEWER chunks with broader context per chunk.


## Step 3: Embed + Store

Embed chunks and persist in ChromaDB.

In [23]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# Use the 500-sized chunks for retrieval testing
chunks = chunks_500

persist_dir = str(BASE_DIR / 'chroma_part6_store')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='part6_rag_collection',
    persist_directory=persist_dir
)

print(f'Embedded and stored {len(chunks)} chunks in Chroma.')
print(f'Persistence directory: {persist_dir}')

Embedded and stored 26 chunks in Chroma.
Persistence directory: c:\Users\leiwang\OneDrive - Stanford Childrens Health\AI-Internship\ai-engineering-bootcamp\rag-vector-databases\chroma_part6_store


## Step 4: Test Retrieval (before LLM wiring)

Run 3 queries and print top-3 chunks with relevance annotations.

In [24]:
queries = [
    'How to Use ChatGPT in Your Job Search Effectively?',
    '5 sample prompts and insights you can use as a starting point to explore careers?',
    'How to Refine Your SCAR Stories?'
]

for q in queries:
    print('=' * 90)
    print(f'Query: {q}')
    results = vectorstore.similarity_search(q, k=3)
    for i, doc in enumerate(results, start=1):
        snippet = doc.page_content.replace('\n', ' ')[:300]
        print()
        print(f'Top {i} chunk:')
        print(snippet)

    # Simple manual annotation helper
    combined = ' '.join(d.page_content.lower() for d in results)
    q_terms = [t for t in q.lower().replace('?', '').split() if len(t) > 3]
    overlap = [t for t in q_terms if t in combined]
    is_relevant = 'Yes' if overlap else 'No'
    why = (
        f'Contains query terms {overlap} in retrieved chunks.'
        if overlap else
        'Little to no term/topic overlap with retrieved chunks.'
    )
    print()
    print(f'Relevant? {is_relevant}')
    print(f'Why: {why}')

Query: How to Use ChatGPT in Your Job Search Effectively?

Top 1 chunk:
Tips on How to Use ChatGPT in Your Job and Internship Search Effectively and Ethically Preparing for Interviews Generate 10 specific interviewquestions based on this jobdescription.Take question one and based on myresume, how would you answer thatquestion with a response that feelsconfident and enga

Top 2 chunk:
Supercharge Your Job Search with AI: Direct Strategies  By: Daniel Lopez, Career Services Advisor at RochesterWorks  I recently read an article stating that only 7% of Americans use ChatGPT daily. It's surprising how  under-adopted such a powerful tool remains. When I teach a class or show an indivi

Top 3 chunk:
Supercharge Your Job Search with AI: Direct Strategies  By: Daniel Lopez, Career Services Advisor at RochesterWorks  I recently read an article stating that only 7% of Americans use ChatGPT daily. It's surprising how  under-adopted such a powerful tool remains. When I teach a class or show an indi

## Step 5: Build the RAG Chain

Wire up `RetrievalQA` with a custom prompt template, run the same 3 queries through the full chain, and print the generated answers.

### How This Chain Works

`RetrievalQA` combines retrieval and answer generation in one step:

1. The retriever searches the vector store for the most relevant chunks.
2. Those retrieved chunks are inserted into `{context}`.
3. The current user query is inserted into `{question}`.
4. The completed prompt is sent to the LLM to generate the final answer.

In this notebook:
- `{context}` = the top matching text chunks from Chroma
- `{question}` = the current query from the `queries` list
- `RetrievalQA` = the wrapper that connects retriever + prompt + LLM

In [28]:
prompt_template = PromptTemplate(
    input_variables=['context', 'question'],
    template=(
        'You are a helpful job-search assistant. Use only the provided context to answer '
        'the question. If the answer is not in the context, say you do not have enough '
        'information from the documents.\n\n'
        'Context:\n{context}\n\n'
        'Question: {question}\n\n'
        'Answer:'
    )
)

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=False,
    chain_type_kwargs={'prompt': prompt_template}
)

for i, q in enumerate(queries, start=1):
    print('=' * 90)
    print(f'Query {i}: {q}')

    retrieved_docs = retriever.invoke(q)
    context_text = '\n\n'.join(doc.page_content for doc in retrieved_docs)
    formatted_prompt = prompt_template.format(context=context_text, question=q)

    if i == 1:
        print('Formatted prompt sent to the model:')
        print(formatted_prompt)
        print('=' * 90)

    response = rag_chain.invoke({'query': q})
    print('Generated answer:')
    print(response['result'])

Query 1: How to Use ChatGPT in Your Job Search Effectively?
Formatted prompt sent to the model:
You are a helpful job-search assistant. Use only the provided context to answer the question. If the answer is not in the context, say you do not have enough information from the documents.

Context:
Tips on How to Use ChatGPT in Your Job and Internship
Search Effectively and Ethically
Preparing for Interviews
Generate 10 specific interviewquestions based on this jobdescription.Take question one and based on myresume, how would you answer thatquestion with a response that feelsconfident and engaging?What are five questions I can ask in aninterview that will give me insight intothe company’s culture andcommitment to DEI?
Sample Prompts to Use
ChatGPT, AI, andthe Job Search

Supercharge Your Job Search with AI: Direct Strategies 
By: Daniel Lopez, Career Services Advisor at RochesterWorks 
I recently read an article stating that only 7% of Americans use ChatGPT daily. It's surprising how 
unde

## Step 6: Evaluate

Create a small eval set with 3 known-answer questions, run them through the RAG pipeline, and score:

- ✅ / ❌ retrieval quality
- ✅ / ❌ faithfulness to the retrieved context
- ✅ / ❌ answer correctness

This is a lightweight heuristic eval based on keyword matching, so it is useful for quick checks but not a substitute for human review.

In [29]:
eval_set = [
    {
        'question': 'How can ChatGPT help someone prepare for interviews?',
        'expected_answer_terms': ['interview', 'questions', 'confident'],
        'expected_context_terms': ['preparing for interviews', 'interview', 'questions']
    },
    {
        'question': 'What is one sample prompt from the documents for exploring careers?',
        'expected_answer_terms': ['what careers fit me', 'careers'],
        'expected_context_terms': ['what careers fit me', 'suggest careers', 'careers']
    },
    {
        'question': 'How can AI help refine SCAR stories?',
        'expected_answer_terms': ['feedback', 'clarity', 'impact'],
        'expected_context_terms': ['scar', 'clarity', 'impact']
    }
]


def passes_threshold(text, expected_terms, minimum_hits=1):
    text_lower = text.lower()
    hits = sum(term.lower() in text_lower for term in expected_terms)
    return hits >= minimum_hits, hits


retrieval_score = 0
faithfulness_score = 0
correctness_score = 0

for i, item in enumerate(eval_set, start=1):
    question = item['question']
    retrieved_docs = retriever.invoke(question)
    context_text = '\n\n'.join(doc.page_content for doc in retrieved_docs)
    answer = rag_chain.invoke({'query': question})['result']

    retrieval_ok, retrieval_hits = passes_threshold(
        context_text,
        item['expected_context_terms'],
        minimum_hits=1
    )

    answer_has_expected, answer_hits = passes_threshold(
        answer,
        item['expected_answer_terms'],
        minimum_hits=1
    )

    grounded_terms = [
        term for term in item['expected_answer_terms']
        if term.lower() in answer.lower() and term.lower() in context_text.lower()
    ]
    faithfulness_ok = bool(grounded_terms) and 'do not have enough information' not in answer.lower()
    correctness_ok = answer_has_expected

    retrieval_score += int(retrieval_ok)
    faithfulness_score += int(faithfulness_ok)
    correctness_score += int(correctness_ok)

    print('=' * 90)
    print(f'Eval {i}: {question}')
    print('Answer:')
    print(answer)
    print()
    print(f"{'✅' if retrieval_ok else '❌'} Retrieved the right chunks")
    print(f"{'✅' if faithfulness_ok else '❌'} Answer is grounded in the context")
    print(f"{'✅' if correctness_ok else '❌'} Answer is correct")
    print(
        f"Details: retrieval hits={retrieval_hits}, answer hits={answer_hits}, "
        f"grounded terms={grounded_terms}"
    )

print('\n' + '=' * 90)
print('Mini Eval Summary')
print(f'Retrieval accuracy: {retrieval_score}/3')
print(f'Faithfulness accuracy: {faithfulness_score}/3')
print(f'Correctness accuracy: {correctness_score}/3')

Eval 1: How can ChatGPT help someone prepare for interviews?
Answer:
ChatGPT can help someone prepare for interviews by generating specific interview questions based on the job description, assisting in crafting confident and engaging responses to those questions, and providing insights into the company’s culture and commitment to diversity, equity, and inclusion (DEI) through suggested questions to ask during the interview.

✅ Retrieved the right chunks
✅ Answer is grounded in the context
✅ Answer is correct
Details: retrieval hits=3, answer hits=3, grounded terms=['interview', 'questions', 'confident']
Eval 2: What is one sample prompt from the documents for exploring careers?
Answer:
One sample prompt for exploring careers is: “I’m a [degree major] student who loves [passions] and is great at [skills]. What careers fit me?”

✅ Retrieved the right chunks
✅ Answer is grounded in the context
✅ Answer is correct
Details: retrieval hits=1, answer hits=2, grounded terms=['careers']
Eval 3